Setup e Bibliotecas

In [0]:
from pyspark.sql.functions import col, year, month, countDistinct, count, sum, round, avg, when
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

Tabela Principal - Comercial

In [0]:
df_pedidos = spark.table("silver.fat_pedido_total")
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_cotacao = spark.table("silver.dim_cotacao_dolar")

# Cruzamento base para a visão comercial
df_base_vendas = df_pedidos.select("id_pedido", "data_pedido") \
    .join(df_itens, "id_pedido") \
    .join(df_produtos, "id_produto") \
    .join(df_cotacao, df_pedidos.data_pedido == df_cotacao.data_cotacao, "left")

df_vendas_comercial = df_base_vendas.withColumn("ano_venda", year("data_pedido")) \
    .withColumn("mes_venda", month("data_pedido")) \
    .groupBy("ano_venda", "mes_venda", "categoria_produto") \
    .agg(
        countDistinct("id_pedido").alias("total_pedidos"),
        count("id_item").alias("qtd_itens_vendidos"),
        round(sum("preco_BRL"), 2).alias("receita_total_brl"),
        # Para garantir a precisão a nível de item, calculamos o USD sobre o item usando a cotação do dia do pedido
        round(sum(col("preco_BRL") / col("valor_cotacao_usd")), 2).alias("receita_total_usd")
    ) \
    .withColumn("ticket_medio_brl", round(col("receita_total_brl") / col("total_pedidos"), 2))

df_vendas_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_vendas_comercial")

In [0]:
print("Verificação da Tabela Comercial")
df_comercial = spark.table("gold.fat_vendas_comercial")
df_comercial.show(3, truncate=False)

Verificação da Tabela Comercial
+---------+---------+----------------------+-------------+------------------+-----------------+-----------------+----------------+
|ano_venda|mes_venda|categoria_produto     |total_pedidos|qtd_itens_vendidos|receita_total_brl|receita_total_usd|ticket_medio_brl|
+---------+---------+----------------------+-------------+------------------+-----------------+-----------------+----------------+
|2017     |6        |informatica_acessorios|219          |261               |37007.08         |11250.77         |168.98          |
|2018     |3        |cama_mesa_banho       |670          |798               |69256.39         |21105.32         |103.37          |
|2017     |3        |cama_mesa_banho       |249          |289               |25773.02         |8239.87          |103.51          |
+---------+---------+----------------------+-------------+------------------+-----------------+-----------------+----------------+
only showing top 3 rows


Rankings Comerciais

In [0]:
df_rank_produtos = df_base_vendas.groupBy("nome_produto", "categoria_produto") \
    .agg(count("id_item").alias("quantidade_vendida"))

df_rank_produtos_validos = df_rank_produtos.filter(col("quantidade_vendida") > 0)

print("Top 5 Produtos MAIS Vendidos:")
display(df_rank_produtos_validos.orderBy(col("quantidade_vendida").desc()).limit(5))

print("Top 5 Produtos MENOS Vendidos:")
display(df_rank_produtos_validos.orderBy(col("quantidade_vendida").asc()).limit(5))

Top 5 Produtos MAIS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


Top 5 Produtos MENOS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1


Tabela Principal - Qualidade

In [0]:
df_avaliacoes = spark.table("silver.fat_avaliacoes_pedidos")
df_vendedores = spark.table("silver.dim_vendedores").select("id_vendedor", "nome_vendedor", col("estado").alias("estado_vendedor"))

df_base_qualidade = df_avaliacoes.join(df_itens, "id_pedido") \
    .join(df_produtos, "id_produto") \
    .join(df_vendedores, "id_vendedor")

df_qualidade_clientes = df_base_qualidade.groupBy("categoria_produto", "nome_vendedor", "estado_vendedor") \
    .agg(
        countDistinct("id_avaliacao").alias("total_avaliacoes"),
        round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        sum(when(col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    ) \
    .withColumn("percentual_satisfacao", round((col("total_avaliacoes_positivas") / col("total_avaliacoes")) * 100, 2))

df_qualidade_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fat_avaliacoes_clientes")

In [0]:
print("\nVerificação da Tabela de Qualidade")
df_qualidade = spark.table("gold.fat_avaliacoes_clientes")
df_qualidade.show(3, truncate=False)


Verificação da Tabela de Qualidade
+---------------------------------+----------------------+---------------+----------------+---------------+--------------------------+--------------------------+---------------------+
|categoria_produto                |nome_vendedor         |estado_vendedor|total_avaliacoes|avaliacao_media|total_avaliacoes_positivas|total_avaliacoes_negativas|percentual_satisfacao|
+---------------------------------+----------------------+---------------+----------------+---------------+--------------------------+--------------------------+---------------------+
|construcao_ferramentas_construcao|Dr. José Câmara       |SP             |2               |3.0            |1                         |1                         |50.0                 |
|utilidades_domesticas            |Mirella Nogueira      |SP             |22              |4.36           |20                        |2                         |90.91                |
|esporte_lazer                    |Sr. Ravi 

Rankings de Qualidade

In [0]:
# Agregações baseadas na regra de desempate (Nota -> Volume decrescente)
rank_prod_qualidade = df_base_qualidade.groupBy("nome_produto").agg(avg("nota_avaliacao").alias("nota_media"), countDistinct("id_avaliacao").alias("volume"))
rank_vend_qualidade = df_base_qualidade.groupBy("nome_vendedor").agg(avg("nota_avaliacao").alias("nota_media"), countDistinct("id_avaliacao").alias("volume"))

print("Produto MAIS bem avaliado:")
display(rank_prod_qualidade.orderBy(col("nota_media").desc(), col("volume").desc()).limit(1))

print("Produto MENOS bem avaliado:")
display(rank_prod_qualidade.orderBy(col("nota_media").asc(), col("volume").desc()).limit(1))

print("Vendedor MAIS bem avaliado:")
display(rank_vend_qualidade.orderBy(col("nota_media").desc(), col("volume").desc()).limit(1))

print("Vendedor MENOS bem avaliado:")
display(rank_vend_qualidade.orderBy(col("nota_media").asc(), col("volume").desc()).limit(1))

Produto MAIS bem avaliado:


nome_produto,nota_media,volume
Carrinho de Controle Remoto Master,5.0,11


Produto MENOS bem avaliado:


nome_produto,nota_media,volume
Luminária Criativa Master,1.0,3


Vendedor MAIS bem avaliado:


nome_vendedor,nota_media,volume
Luiz Otávio Abreu,5.0,31


Vendedor MENOS bem avaliado:


nome_vendedor,nota_media,volume
Sra. Fernanda Santos,1.0,3
